# Credit Card Fraud Decision Agent
## Stage 1 — Dataset inspection and evidence definitions

This notebook begins with the raw historical dataset. Stage 1 only verifies the data and creates five binary evidence variables. Priors, likelihoods, Bayesian updates, and decisions are added in later stages.

In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/credit_card_fraud_10k.csv')
df = pd.read_csv(DATA_PATH)

print(f'Rows: {len(df):,}')
print('Columns:', list(df.columns))
print('Missing values:', int(df.isna().sum().sum()))

Rows: 10,000
Columns: ['transaction_id', 'amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age', 'is_fraud']
Missing values: 0


In [5]:
REQUIRED_COLUMNS = [
    'transaction_hour',
    'foreign_transaction',
    'location_mismatch',
    'device_trust_score',
    'velocity_last_24h',
    'is_fraud',
]

assert set(REQUIRED_COLUMNS).issubset(df.columns)
assert df['transaction_hour'].between(0, 23).all()
assert df['foreign_transaction'].isin([0, 1]).all()
assert df['location_mismatch'].isin([0, 1]).all()
assert df['is_fraud'].isin([0, 1]).all()
print('Required columns and basic ranges verified.')

Required columns and basic ranges verified.


### Five binary evidence variables

The thresholds below are design assumptions for this beginner-friendly V1. They are not probabilities learned from the dataset.

In [6]:
df['EARLY_HOUR'] = df['transaction_hour'] <= 5
df['FOREIGN_TRANSACTION'] = df['foreign_transaction'] == 1
df['LOCATION_MISMATCH'] = df['location_mismatch'] == 1
df['LOW_DEVICE_TRUST'] = df['device_trust_score'] < 50
df['HIGH_VELOCITY'] = df['velocity_last_24h'] >= 3

EVIDENCE_COLUMNS = [
    'EARLY_HOUR', 'FOREIGN_TRANSACTION', 'LOCATION_MISMATCH',
    'LOW_DEVICE_TRUST', 'HIGH_VELOCITY'
]

print('Evidence columns:', EVIDENCE_COLUMNS)
print(df[EVIDENCE_COLUMNS].sum().to_string())

Evidence columns: ['EARLY_HOUR', 'FOREIGN_TRANSACTION', 'LOCATION_MISMATCH', 'LOW_DEVICE_TRUST', 'HIGH_VELOCITY']
EARLY_HOUR             2455
FOREIGN_TRANSACTION     978
LOCATION_MISMATCH       857
LOW_DEVICE_TRUST       3359
HIGH_VELOCITY          3266


## Stage 2 — Historical priors and likelihoods

Every probability below is calculated from row counts. No expected value is hardcoded.

In [7]:
state_counts = df['is_fraud'].value_counts().sort_index()
total_transactions = len(df)
priors = {
    'LEGITIMATE': state_counts[0] / total_transactions,
    'FRAUDULENT': state_counts[1] / total_transactions,
}

print('Class counts:')
print(f"LEGITIMATE: {state_counts[0]:,}")
print(f"FRAUDULENT: {state_counts[1]:,}")
print('Priors:', priors)
assert abs(sum(priors.values()) - 1.0) < 1e-12

Class counts:
LEGITIMATE: 9,849
FRAUDULENT: 151
Priors: {'LEGITIMATE': np.float64(0.9849), 'FRAUDULENT': np.float64(0.0151)}


In [8]:
likelihood_rows = []
for evidence in EVIDENCE_COLUMNS:
    for state_name, state_value in [('LEGITIMATE', 0), ('FRAUDULENT', 1)]:
        denominator = int((df['is_fraud'] == state_value).sum())
        numerator = int(((df['is_fraud'] == state_value) & df[evidence]).sum())
        probability_true = numerator / denominator
        likelihood_rows.append({
            'evidence': evidence,
            'state': state_name,
            'true_count': numerator,
            'state_count': denominator,
            'P(evidence=True | state)': probability_true,
            'P(evidence=False | state)': 1 - probability_true,
        })

likelihood_table = pd.DataFrame(likelihood_rows)
display(likelihood_table)
assert likelihood_table['P(evidence=True | state)'].between(0, 1).all()
assert likelihood_table['P(evidence=False | state)'].between(0, 1).all()

,evidence,state,true_count,state_count,P(evidence=True | state),P(evidence=False | state)
0,EARLY_HOUR,LEGITIMATE,2331,9849,0.236674,0.763326
1,EARLY_HOUR,FRAUDULENT,124,151,0.821192,0.178808
2,FOREIGN_TRANSACTION,LEGITIMATE,896,9849,0.090974,0.909026
3,FOREIGN_TRANSACTION,FRAUDULENT,82,151,0.543046,0.456954
4,LOCATION_MISMATCH,LEGITIMATE,785,9849,0.079704,0.920296
5,LOCATION_MISMATCH,FRAUDULENT,72,151,0.476821,0.523179
6,LOW_DEVICE_TRUST,LEGITIMATE,3230,9849,0.327952,0.672048
7,LOW_DEVICE_TRUST,FRAUDULENT,129,151,0.854305,0.145695
8,HIGH_VELOCITY,LEGITIMATE,3178,9849,0.322672,0.677328
9,HIGH_VELOCITY,FRAUDULENT,88,151,0.582781,0.417219


## Stage 3 — Sequential Bayesian belief update

For each observed evidence item, multiply the current belief by the matching likelihood. Then divide by the total so the beliefs sum to 1.

In [9]:
def likelihood_for(evidence_name, observed_value, state_name):
    row = likelihood_table[
        (likelihood_table['evidence'] == evidence_name)
        & (likelihood_table['state'] == state_name)
    ].iloc[0]
    if observed_value:
        return row['P(evidence=True | state)']
    return row['P(evidence=False | state)']


def update_belief(current_belief, evidence_name, observed_value):
    likelihood_legitimate = likelihood_for(evidence_name, observed_value, 'LEGITIMATE')
    likelihood_fraudulent = likelihood_for(evidence_name, observed_value, 'FRAUDULENT')

    unnormalized_legitimate = current_belief['LEGITIMATE'] * likelihood_legitimate
    unnormalized_fraudulent = current_belief['FRAUDULENT'] * likelihood_fraudulent
    normalizer = unnormalized_legitimate + unnormalized_fraudulent

    updated_belief = {
        'LEGITIMATE': unnormalized_legitimate / normalizer,
        'FRAUDULENT': unnormalized_fraudulent / normalizer,
    }
    assert abs(sum(updated_belief.values()) - 1.0) < 1e-12
    return updated_belief, {
        'likelihood_legitimate': likelihood_legitimate,
        'likelihood_fraudulent': likelihood_fraudulent,
        'unnormalized_legitimate': unnormalized_legitimate,
        'unnormalized_fraudulent': unnormalized_fraudulent,
        'normalizer': normalizer,
    }

In [10]:
belief = priors.copy()
print('Prior:', belief)

belief, early_trace = update_belief(belief, 'EARLY_HOUR', True)
print('After EARLY_HOUR=True:')
print(early_trace)
print('Posterior:', belief)

belief, foreign_trace = update_belief(belief, 'FOREIGN_TRANSACTION', False)
print('After FOREIGN_TRANSACTION=False:')
print(foreign_trace)
print('Posterior:', belief)
assert abs(sum(belief.values()) - 1.0) < 1e-12

Prior: {'LEGITIMATE': np.float64(0.9849), 'FRAUDULENT': np.float64(0.0151)}
After EARLY_HOUR=True:
{'likelihood_legitimate': np.float64(0.23667377398720682), 'likelihood_fraudulent': np.float64(0.8211920529801324), 'unnormalized_legitimate': np.float64(0.2331), 'unnormalized_fraudulent': np.float64(0.0124), 'normalizer': np.float64(0.2455)}
Posterior: {'LEGITIMATE': np.float64(0.9494908350305499), 'FRAUDULENT': np.float64(0.0505091649694501)}
After FOREIGN_TRANSACTION=False:
{'likelihood_legitimate': np.float64(0.9090262970859986), 'likelihood_fraudulent': np.float64(0.45695364238410596), 'unnormalized_legitimate': np.float64(0.8631121378849135), 'unnormalized_fraudulent': np.float64(0.023080346906569914), 'normalizer': np.float64(0.8861924847914834)}
Posterior: {'LEGITIMATE': np.float64(0.9739556052407727), 'FRAUDULENT': np.float64(0.026044394759227284)}


## Stage 4 — Cost-based decision policy

The policy compares the expected cost of APPROVE and BLOCK. If the costs are close, it asks for more evidence when possible or sends the case to HUMAN_REVIEW when no evidence remains.

In [11]:
POLICY_A_COSTS = {
    'APPROVE': {'LEGITIMATE': 0, 'FRAUDULENT': 10},
    'BLOCK': {'LEGITIMATE': 6, 'FRAUDULENT': 0},
}

POLICY_B_COSTS = {
    'APPROVE': {'LEGITIMATE': 0, 'FRAUDULENT': 20},
    'BLOCK': {'LEGITIMATE': 6, 'FRAUDULENT': 0},
}

# DESIGN ASSUMPTION: relative-cost gap of 0.5 is considered too close.
UNCERTAINTY_MARGIN = 0.5

In [12]:
def expected_decision_costs(belief, policy_costs):
    approve_cost = (
        belief['LEGITIMATE'] * policy_costs['APPROVE']['LEGITIMATE']
        + belief['FRAUDULENT'] * policy_costs['APPROVE']['FRAUDULENT']
    )
    block_cost = (
        belief['LEGITIMATE'] * policy_costs['BLOCK']['LEGITIMATE']
        + belief['FRAUDULENT'] * policy_costs['BLOCK']['FRAUDULENT']
    )
    return {'APPROVE': approve_cost, 'BLOCK': block_cost}


def choose_action(belief, unused_evidence, policy_costs):
    costs = expected_decision_costs(belief, policy_costs)
    cost_gap = abs(costs['APPROVE'] - costs['BLOCK'])

    if cost_gap <= UNCERTAINTY_MARGIN:
        if unused_evidence:
            return 'GET_MORE_EVIDENCE', costs, cost_gap
        return 'HUMAN_REVIEW', costs, cost_gap

    action = min(costs, key=costs.get)
    return action, costs, cost_gap

In [13]:
costs = expected_decision_costs(belief, POLICY_A_COSTS)
action, costs, cost_gap = choose_action(
    belief,
    unused_evidence=['LOCATION_MISMATCH', 'LOW_DEVICE_TRUST', 'HIGH_VELOCITY'],
    policy_costs=POLICY_A_COSTS,
)

print('Current posterior:', belief)
print('Expected APPROVE cost:', costs['APPROVE'])
print('Expected BLOCK cost:', costs['BLOCK'])
print('Cost gap:', cost_gap)
print('Selected action:', action)

Current posterior: {'LEGITIMATE': np.float64(0.9739556052407727), 'FRAUDULENT': np.float64(0.026044394759227284)}
Expected APPROVE cost: 0.26044394759227285
Expected BLOCK cost: 5.843733631444636
Cost gap: 5.583289683852364
Selected action: APPROVE


## Stage 5 — Complete sequential agent loop

The agent initially sees only EARLY_HOUR and FOREIGN_TRANSACTION. If the policy requests more evidence, it reveals the remaining evidence in the fixed V1 order. The hidden fraud label is not used while deciding.

In [14]:
INITIAL_EVIDENCE_ORDER = ['EARLY_HOUR', 'FOREIGN_TRANSACTION']
ADDITIONAL_EVIDENCE_ORDER = [
    'LOCATION_MISMATCH',
    'LOW_DEVICE_TRUST',
    'HIGH_VELOCITY',
]


def evidence_from_transaction(transaction):
    return {
        'EARLY_HOUR': transaction['transaction_hour'] <= 5,
        'FOREIGN_TRANSACTION': transaction['foreign_transaction'] == 1,
        'LOCATION_MISMATCH': transaction['location_mismatch'] == 1,
        'LOW_DEVICE_TRUST': transaction['device_trust_score'] < 50,
        'HIGH_VELOCITY': transaction['velocity_last_24h'] >= 3,
    }

In [15]:
def run_agent(transaction, policy_costs, hidden_label=None, verbose=True):
    all_evidence = evidence_from_transaction(transaction)
    evidence_order = INITIAL_EVIDENCE_ORDER + ADDITIONAL_EVIDENCE_ORDER
    belief = priors.copy()
    collected_evidence = []
    trace = []

    for evidence_name in evidence_order:
        if evidence_name not in INITIAL_EVIDENCE_ORDER and trace:
            previous_action = trace[-1]['action']
            if previous_action != 'GET_MORE_EVIDENCE':
                break

        observed_value = all_evidence[evidence_name]
        belief, update_trace = update_belief(
            belief, evidence_name, observed_value
        )
        collected_evidence.append(evidence_name)
        unused_evidence = [
            name for name in evidence_order
            if name not in collected_evidence
        ]
        action, costs, cost_gap = choose_action(
            belief, unused_evidence, policy_costs
        )
        step = {
            'evidence_name': evidence_name,
            'observed_value': observed_value,
            'belief': belief.copy(),
            'update_trace': update_trace,
            'costs': costs,
            'cost_gap': cost_gap,
            'action': action,
            'collected_evidence': collected_evidence.copy(),
            'unused_evidence': unused_evidence,
        }
        trace.append(step)

        if verbose:
            print(f"Evidence: {evidence_name} = {observed_value}")
            print(f"Belief: {belief}")
            print(f"Expected costs: {costs}")
            print(f"Action: {action}")
            print()

        if action in ['APPROVE', 'BLOCK', 'HUMAN_REVIEW']:
            break

    final_step = trace[-1]
    result = {
        'final_action': final_step['action'],
        'final_belief': final_step['belief'],
        'collected_evidence': final_step['collected_evidence'],
        'trace': trace,
    }
    if hidden_label is not None:
        result['hidden_label_unused_during_decision'] = hidden_label
    return result

In [16]:
example_transaction = {
    'transaction_hour': 3,
    'foreign_transaction': 0,
    'location_mismatch': 1,
    'device_trust_score': 35,
    'velocity_last_24h': 1,
}

example_evidence = evidence_from_transaction(example_transaction)
print('All derived evidence:', example_evidence)
print('Initially revealed:', INITIAL_EVIDENCE_ORDER)
print()
policy_a_result = run_agent(example_transaction, POLICY_A_COSTS)
policy_b_result = run_agent(example_transaction, POLICY_B_COSTS)

print('Policy A final action:', policy_a_result['final_action'])
print('Policy B final action:', policy_b_result['final_action'])
print('Policy A final belief:', policy_a_result['final_belief'])
print('Policy B final belief:', policy_b_result['final_belief'])

All derived evidence: {'EARLY_HOUR': True, 'FOREIGN_TRANSACTION': False, 'LOCATION_MISMATCH': True, 'LOW_DEVICE_TRUST': True, 'HIGH_VELOCITY': False}
Initially revealed: ['EARLY_HOUR', 'FOREIGN_TRANSACTION']

Evidence: EARLY_HOUR = True
Belief: {'LEGITIMATE': np.float64(0.9494908350305499), 'FRAUDULENT': np.float64(0.0505091649694501)}
Expected costs: {'APPROVE': np.float64(0.505091649694501), 'BLOCK': np.float64(5.696945010183299)}
Action: APPROVE

Final action: APPROVE
Final belief: {'LEGITIMATE': np.float64(0.9494908350305499), 'FRAUDULENT': np.float64(0.0505091649694501)}
Evidence collected: ['EARLY_HOUR']
